# Study 09 · 05 · E4 · Is the climate penalty a calibration artefact?

On the six seasons, b3's Brier penalty against history+geography came from miscalibration, not from discrimination. The NB2 dispersion α is fitted **in-sample** on the training fit, which can make the model with more predictors overconfident before Platt scaling. Three calibration variants, applied to both arms on the same raw model means:

1. **current:** in-sample α, then chronological Platt (the paper);
2. **out-of-sample α:** α refitted on earlier blocks' (μ, cases) with labels closed by each block's cut-off, then chronological Platt;
3. **isotonic:** chronological isotonic recalibration of the current raw probability (the PAV method behind CORP).

For each variant: the b3 vs history+geography contrast (18 seasons, pooled) and the CORP split.

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt

import experiments as ex

FIG = STUDY / "figures"
RES = c.OUTPUT / "resultados"
FIG.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)
base = pd.read_csv(c.OUTPUT / "baselines.csv.gz", dtype={"ubigeo": str}, parse_dates=["semana_inicio", "origen_cierre"])
frames = {arm: c.read_arm(arm, columns=ex.CALIBRATION_COLUMNS) for arm in ("b3_obs", "historia_geo")}

In [2]:
variants, alpha_records = {}, []
for arm, frame in frames.items():
    parts = []
    for seed, f in frame.groupby("semilla"):
        f = f.reset_index(drop=True)
        raw_oos, record = ex.out_of_sample_alpha(f)
        alpha_records.append(record.assign(brazo=arm, semilla=seed))
        parts.append(f.assign(p_current=f.probabilidad_calibrada,
                              p_oos_alpha=ex.recalibrate(f, raw_oos, "platt"),
                              p_isotonic=ex.recalibrate(f, f.probabilidad_cruda.to_numpy(float), "isotonica")))
    variants[arm] = pd.concat(parts, ignore_index=True)
alphas = pd.concat(alpha_records)
alphas.to_csv(RES / "e4_alfas.csv", index=False)
alphas.groupby(["brazo", "horizonte"]).apply(lambda d: pd.Series({
    "in-sample alpha (median)": d.alpha_in_sample.median(), "out-of-sample alpha (median)": d.alpha_used.median(),
    "share of blocks with out-of-sample alpha": (d.source == "out_of_sample").mean()}), include_groups=False).round(3)

in-sample alpha (median)  out-of-sample alpha (median)  share of blocks with out-of-sample alpha
brazo        horizonte                                                                                                  
b3_obs       2                             1.522                         1.726                                      0.95
             3                             2.452                         2.958                                      0.95
             4                             3.420                         4.493                                      0.95
historia_geo 2                             1.553                         1.697                                      0.95
             3                             2.561                         2.904                                      0.95
             4                             3.665                         4.436                                      0.95

In [3]:
rows = []
for variant in ("p_current", "p_oos_alpha", "p_isotonic"):
    a = ex.sse(variants["b3_obs"], variant)
    b = ex.sse(variants["historia_geo"], variant)
    rows.append(ex.contrast(a, b).assign(variant=variant))
calibration_contrast = pd.concat(rows)[["variant", "horizonte", "brier_a", "brier_b", "cambio_agrupado_%", "ic95_agrupado", "cambio_igual_peso_%", "temporadas_a_mejor"]]
calibration_contrast.to_csv(RES / "e4_contrastes.csv", index=False)
calibration_contrast.round(4)

,variant,horizonte,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor
0,p_current,2,0.0282,0.0283,-0.4300,"(-1.46, 0.37)",-1.3127,10
1,p_current,3,0.0322,0.0323,-0.2760,"(-1.89, 0.87)",-1.3944,10
2,p_current,4,0.0356,0.0356,0.0416,"(-2.31, 1.7)",-1.1533,10
0,p_oos_alpha,2,0.0288,0.0289,-0.1819,"(-1.45, 0.99)",-0.9628,10
1,p_oos_alpha,3,0.0332,0.0334,-0.6367,"(-2.54, 1.02)",-1.3674,11
2,p_oos_alpha,4,0.0370,0.0375,-1.0979,"(-4.23, 1.27)",-1.7943,11
0,p_isotonic,2,0.0274,0.0275,-0.2357,"(-1.38, 0.8)",-0.6241,9
1,p_isotonic,3,0.0311,0.0314,-0.7412,"(-2.63, 0.75)",-1.6592,11
2,p_isotonic,4,0.0344,0.0346,-0.6665,"(-3.26, 1.34)",-1.5656,9


In [4]:
corp_rows = []
for arm, f in variants.items():
    d = ex.seasons_only(f)
    for variant in ("p_current", "p_oos_alpha", "p_isotonic"):
        for (seed, h), g in d.groupby(["semilla", "horizonte"]):
            corp_rows.append({"brazo": arm, "variant": variant, "semilla": seed, "horizonte": h, **c.corp(g[variant], g.brote)})
corp_variants = pd.DataFrame(corp_rows).groupby(["variant", "horizonte", "brazo"])[["brier", "mcb", "dsc"]].median().unstack("brazo")
corp_variants.to_csv(RES / "e4_corp.csv")
corp_variants.round(5)

brier                   mcb                   dsc             
brazo                   b3_obs historia_geo   b3_obs historia_geo   b3_obs historia_geo
variant     horizonte                                                                  
p_current   2          0.02821      0.02833  0.00087      0.00093  0.04266      0.04259
            3          0.03217      0.03226  0.00115      0.00092  0.03896      0.03865
            4          0.03563      0.03561  0.00125      0.00093  0.03564      0.03532
p_isotonic  2          0.02744      0.02752  0.00025      0.00029  0.04280      0.04277
            3          0.03111      0.03136  0.00034      0.00037  0.03921      0.03900
            4          0.03437      0.03460  0.00038      0.00052  0.03604      0.03592
p_oos_alpha 2          0.02882      0.02888  0.00078      0.00097  0.04194      0.04207
            3          0.03317      0.03338  0.00114      0.00130  0.03793      0.03791
            4          0.03702      0.03749  0.00136      0.00160  0.03433      0.03412